# Limpeza de `campanha_ago26.csv`

**Objetivo:** gerar a tabela da campanha de 10/08/2026 pronta para análise em `dados/processados/campanha_limpa.parquet`.

O diagnóstico (notebook 11) não achou nada para tirar: nenhuma conta de teste, cópia, duplicata ou valor fora do domínio.
**As 14.120 linhas ficam.** A limpeza converte os tipos e cria as marcas pedidas pelas decisões de 07/11/2026:

| Passo | O que faz | Origem |
|---|---|---|
| 1 | `enviado_em` vira data e hora (Brasília); o vazio do controle vira ausente | tipos; fuso tratado como Brasília (decisão de 07/11/2026) |
| 2 | `canal`, `titulo_mensagem` e `oferta` vazios viram ausente; engajamento vira booleano | tipos |
| 3 | `tratado`, `plano_no_envio` e `canal_cobranca_no_envio` | intenção de tratar (decisão de 07/11/2026) |
| 4 | `assinou_depois_do_envio`, `saiu_antes_do_envio` e `fora_da_janela_do_envio` | sensibilidade sem os 48 (decisão de 07/11/2026) |
| 5 | `aniversario_anual` e `aniversario_na_janela` | anuais com aniversário entre 10/08 e 30/09 (decisão de 07/11/2026) |

**Atenção ao usar:** compare os grupos inteiros com o controle (intenção de tratar). `abriu_push`, `abriu_email` e `clicou`
não têm data e são consequência da mensagem: servem para descrever a atenção, não como preditor nem como filtro do efeito.
O cupom não aparece nas cobranças: o custo do cupom só entra na conta de retorno depois de o dono do dado confirmar o resgate.
A tabela não traz desfecho (saída, renovação desligada): ele vem dos eventos de assinatura, na análise.

Nada em `dados/brutos/` é alterado: o hash dos arquivos é conferido no início e no fim.
Os passos vão para `saidas/limpeza_campanha_ago26_registro.csv`.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
DIA = pd.Timestamp("2026-08-10")
ENVIO = pd.Timestamp("2026-08-10 10:00:00")
CORTE = pd.Timestamp("2026-09-30")
GRUPOS = ["controle", "conteudo", "desafio", "cupom"]

ARQUIVOS = ["campanha_ago26.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

registro = []

def registrar(passo, regra, df, alteradas=0):
    anterior = registro[-1]["linhas"] if registro else len(df)
    registro.append({"passo": passo, "regra": regra, "linhas": len(df), "removidas": anterior - len(df),
                     "alteradas": int(alteradas)})
    print(f"{passo}: {len(df):,} linhas ({anterior - len(df):,} removidas, {int(alteradas):,} alteradas)")
    return df

## 0. Leitura

Tudo como texto (`dtype=str, keep_default_na=False`): o arquivo chega como foi gravado e vazio fica `""`.

In [2]:
campanha = pd.read_csv(BRUTOS / "campanha_ago26.csv", dtype=str, keep_default_na=False)
assert len(campanha) == 14_120   # cabeçalho da seção 05 do dicionário
assert campanha["user_id"].is_unique
assert set(campanha["grupo"]) == set(GRUPOS)

base = registrar("0. Arquivo bruto", "campanha_ago26.csv sem filtro", campanha.copy())

0. Arquivo bruto: 14,120 linhas (0 removidas, 0 alteradas)


## 1. Data e hora do envio

Um único formato e um único valor nos grupos tratados (`2026-08-10 10:00:00`), sem fuso: tratado como horário de Brasília.

In [3]:
controle = base["grupo"] == "controle"
assert (base.loc[controle, "enviado_em"] == "").all()
assert (base.loc[~controle, "enviado_em"] == "2026-08-10 10:00:00").all()
base["enviado_em"] = pd.to_datetime(base["enviado_em"].replace("", None), format="%Y-%m-%d %H:%M:%S")
assert base["enviado_em"].isna().sum() == controle.sum()

base = registrar("1. Data e hora do envio", "enviado_em como data e hora (Brasília); vazio do controle vira ausente", base,
                 (~controle).sum())

1. Data e hora do envio: 14,120 linhas (0 removidas, 10,541 alteradas)


## 2. Textos e engajamento

`canal`, `titulo_mensagem` e `oferta` vazios viram ausente (vazio estrutural: controle sem mensagem; oferta só no cupom).
Engajamento de texto (`True`/`False`) para booleano; no controle fica `False`, porque ninguém ali recebeu nada.

In [4]:
vazios = {col: int((base[col] == "").sum()) for col in ["canal", "titulo_mensagem", "oferta"]}
print("vazios que viram ausente:", vazios)
for col in ["canal", "titulo_mensagem", "oferta"]:
    base[col] = base[col].where(base[col] != "", None)
assert base.loc[controle, ["canal", "titulo_mensagem", "oferta"]].isna().all().all()
assert base.loc[~controle, ["canal", "titulo_mensagem"]].notna().all().all()
assert (base["oferta"].notna() == (base["grupo"] == "cupom")).all()

ENGAJAMENTO = ["abriu_push", "abriu_email", "clicou"]
for col in ENGAJAMENTO:
    assert set(base[col]) <= {"True", "False"}, col
    base[col] = base[col].map({"True": True, "False": False}).astype(bool)
assert not base.loc[controle, ENGAJAMENTO].any().any()
assert not (base["clicou"] & ~(base["abriu_push"] | base["abriu_email"])).any()
display(base.groupby("grupo")[ENGAJAMENTO].mean().mul(100).round(1).loc[GRUPOS])

base = registrar("2. Textos e engajamento", "vazios estruturais viram ausente; engajamento como booleano", base,
                 sum(vazios.values()))

vazios que viram ausente: {'canal': 3579, 'titulo_mensagem': 3579, 'oferta': 10648}


,abriu_push,abriu_email,clicou
grupo,,,
controle,0.0,0.0,0.0
conteudo,32.0,18.0,14.9
desafio,30.6,18.4,14.3
cupom,31.6,19.0,16.4


2. Textos e engajamento: 14,120 linhas (0 removidas, 17,806 alteradas)


## 3. Tratamento, plano e canal no envio

`tratado` marca os três grupos com mensagem. Plano e canal de cobrança vêm do log de eventos (cada assinatura tem um único
plano em todo o log); a tabela limpa de assinaturas cobre quem não tem evento com plano.

In [5]:
assinaturas = pd.read_parquet(PROCESSADOS / "assinaturas_limpa.parquet",
                              columns=["user_id", "assinatura_id", "plano_atual", "canal_cobranca", "inicio", "data_fim"])
eventos = pd.read_parquet(PROCESSADOS / "eventos_assinatura_limpo.parquet",
                          columns=["user_id", "assinatura_id", "data_br", "evento", "plano", "canal_cobranca"])

base = base.merge(assinaturas, on="user_id", how="left", validate="one_to_one")
assert base["assinatura_id"].notna().all()

com_plano = eventos[eventos["plano"].notna() & (eventos["plano"] != "")]
por_assinatura = com_plano.groupby("assinatura_id").agg(planos=("plano", "nunique"), plano=("plano", "first"),
                                                         canais=("canal_cobranca", "nunique"), canal=("canal_cobranca", "first"))
assert (por_assinatura["planos"] == 1).all() and (por_assinatura["canais"] == 1).all()
base["plano_no_envio"] = base["assinatura_id"].map(por_assinatura["plano"]).fillna(base["plano_atual"])
base["canal_cobranca_no_envio"] = base["assinatura_id"].map(por_assinatura["canal"]).fillna(base["canal_cobranca"])
assert (base["plano_no_envio"] == base["plano_atual"]).all()

base["tratado"] = base["grupo"] != "controle"
OFERTA = {"mensal": "30pct_3_meses", "anual": "1_mes_gratis"}
cupom = base["grupo"] == "cupom"
assert (base.loc[cupom, "oferta"] == base.loc[cupom, "plano_no_envio"].map(OFERTA)).all()
display(pd.crosstab(base["grupo"], base["plano_no_envio"]).loc[GRUPOS])

base = registrar("3. Tratamento, plano e canal no envio", "tratado; plano_no_envio e canal_cobranca_no_envio pelo log", base)

plano_no_envio,anual,mensal
grupo,,
controle,2139,1440
conteudo,2091,1463
desafio,2059,1456
cupom,2052,1420


3. Tratamento, plano e canal no envio: 14,120 linhas (0 removidas, 0 alteradas)


## 4. Quem não pagava na hora do envio

O sorteio usou o dia 10/08 inteiro. Marca quem assinou às 10h ou depois e quem expirou antes das 10h, em horário de
Brasília. As linhas ficam (intenção de tratar); a marca serve para a análise de sensibilidade nos quatro grupos.

In [6]:
ini = eventos[eventos["evento"] == "assinatura_iniciada"].groupby("user_id")["data_br"].min()
fim = eventos[eventos["evento"] == "assinatura_expirada"].groupby("user_id")["data_br"].min()
base["assinou_depois_do_envio"] = (base["inicio"] == DIA) & base["user_id"].map(ini).ge(ENVIO)
base["saiu_antes_do_envio"] = (base["data_fim"] == DIA) & base["user_id"].map(fim).lt(ENVIO)
base["fora_da_janela_do_envio"] = base["assinou_depois_do_envio"] | base["saiu_antes_do_envio"]
display(base.groupby("grupo")[["assinou_depois_do_envio", "saiu_antes_do_envio", "fora_da_janela_do_envio"]].sum().loc[GRUPOS])
assert base["assinou_depois_do_envio"].sum() == 41 and base["saiu_antes_do_envio"].sum() == 7
assert base["fora_da_janela_do_envio"].sum() == 48

base = registrar("4. Fora da janela do envio", "assinou às 10h ou depois, ou expirou antes das 10h de 10/08", base)

,assinou_depois_do_envio,saiu_antes_do_envio,fora_da_janela_do_envio
grupo,,,
controle,7,3,10
conteudo,13,0,13
desafio,8,1,9
cupom,13,3,16


4. Fora da janela do envio: 14,120 linhas (0 removidas, 0 alteradas)


## 5. Aniversário dos anuais

Do envio ao corte são 51 dias: no anual, só quem faz aniversário nesse intervalo pode sair por não renovar.
`aniversario_anual` é o primeiro aniversário da assinatura a partir de 10/08/2026 (nunca o próprio início);
`aniversario_na_janela` marca os que caem até 30/09/2026 com a assinatura ainda existente no aniversário.

In [7]:
anual = base["plano_no_envio"] == "anual"
aniv = pd.Series(pd.NaT, index=base.index, dtype="datetime64[ns]")
aniv[anual] = [i + pd.DateOffset(years=max(1, DIA.year - i.year)) for i in base.loc[anual, "inicio"]]
antes = anual & (aniv < DIA)
aniv[antes] = [a + pd.DateOffset(years=1) for a in aniv[antes]]
base["aniversario_anual"] = aniv
base["aniversario_na_janela"] = anual & (aniv <= CORTE) & (base["data_fim"].isna() | (base["data_fim"] >= aniv))
display(base[anual].groupby("grupo")["aniversario_na_janela"].agg(["size", "sum"]).loc[GRUPOS]
        .rename(columns={"size": "anuais", "sum": "aniversário na janela"}))
assert base["aniversario_na_janela"].sum() == 1_162
assert anual.sum() == 8_341

base = registrar("5. Aniversário dos anuais", "aniversario_anual; aniversario_na_janela até 30/09/2026", base)

,anuais,aniversário na janela
grupo,,
controle,2139,305
conteudo,2091,273
desafio,2059,303
cupom,2052,281


5. Aniversário dos anuais: 14,120 linhas (0 removidas, 0 alteradas)


## 6. Conferências

In [8]:
usuarios = pd.read_parquet(PROCESSADOS / "usuarios_limpa.parquet", columns=["user_id"])
conferencias = {
    "linhas": (len(base), 14_120),
    "user_id repetido": (int(base["user_id"].duplicated().sum()), 0),
    "contas fora de usuarios_limpa": (int((~base["user_id"].isin(usuarios["user_id"])).sum()), 0),
    "contas sem assinatura limpa": (int(base["assinatura_id"].isna().sum()), 0),
    "controle": (int((base["grupo"] == "controle").sum()), 3_579),
    "controle com envio ou engajamento": (int((~base["tratado"] & (base["enviado_em"].notna() | base[ENGAJAMENTO].any(axis=1))).sum()), 0),
    "oferta diferente do plano no envio": (int((cupom & (base["oferta"] != base["plano_no_envio"].map(OFERTA))).sum()), 0),
    "fora da janela do envio": (int(base["fora_da_janela_do_envio"].sum()), 48),
    "anuais com aniversário na janela": (int(base["aniversario_na_janela"].sum()), 1_162),
}
tabela = pd.DataFrame(conferencias, index=["obtido", "esperado"]).T
tabela["ok"] = tabela["obtido"] == tabela["esperado"]
display(tabela)
assert tabela["ok"].all()

,obtido,esperado,ok
linhas,14120,14120,True
user_id repetido,0,0,True
contas fora de usuarios_limpa,0,0,True
contas sem assinatura limpa,0,0,True
controle,3579,3579,True
controle com envio ou engajamento,0,0,True
oferta diferente do plano no envio,0,0,True
fora da janela do envio,48,48,True
anuais com aniversário na janela,1162,1162,True


## 7. Saídas e integridade dos brutos

In [9]:
COLUNAS = ["user_id", "assinatura_id", "grupo", "tratado", "enviado_em", "canal", "titulo_mensagem", "oferta",
           "abriu_push", "abriu_email", "clicou", "plano_no_envio", "canal_cobranca_no_envio",
           "assinou_depois_do_envio", "saiu_antes_do_envio", "fora_da_janela_do_envio",
           "aniversario_anual", "aniversario_na_janela"]
limpa = base[COLUNAS].reset_index(drop=True)
for col in limpa.columns[limpa.dtypes == object]:
    limpa[col] = limpa[col].astype(object).where(limpa[col].notna(), None)   # ausente sempre como None

PROCESSADOS.mkdir(parents=True, exist_ok=True)
destino = PROCESSADOS / "campanha_limpa.parquet"
limpa.to_parquet(destino, index=False)

relida = pd.read_parquet(destino)
pd.testing.assert_frame_equal(relida, limpa)

pd.DataFrame(registro).to_csv(SAIDAS / "limpeza_campanha_ago26_registro.csv", index=False)

print(f"{destino.relative_to(ROOT)}: {len(limpa):,} linhas x {limpa.shape[1]} colunas")
display(limpa.dtypes.rename("tipo").to_frame().assign(ausentes=limpa.isna().sum()))
display(pd.DataFrame(registro))

hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados\processados\campanha_limpa.parquet: 14,120 linhas x 18 colunas


,tipo,ausentes
user_id,object,0
assinatura_id,object,0
grupo,object,0
tratado,bool,0
enviado_em,datetime64[ns],3579
canal,object,3579
titulo_mensagem,object,3579
oferta,object,10648
abriu_push,bool,0
abriu_email,bool,0


,passo,regra,linhas,removidas,alteradas
0,0. Arquivo bruto,campanha_ago26.csv sem filtro,14120,0,0
1,1. Data e hora do envio,enviado_em como data e hora (Brasília); vazio ...,14120,0,10541
2,2. Textos e engajamento,vazios estruturais viram ausente; engajamento ...,14120,0,17806
3,"3. Tratamento, plano e canal no envio",tratado; plano_no_envio e canal_cobranca_no_en...,14120,0,0
4,4. Fora da janela do envio,"assinou às 10h ou depois, ou expirou antes das...",14120,0,0
5,5. Aniversário dos anuais,aniversario_anual; aniversario_na_janela até 3...,14120,0,0


dados/brutos/ inalterado: True
